In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set.")

PYTORCH_CUDA_ALLOC_CONF set.


In [2]:
import sys
import subprocess

packages = [
    "langchain==1.3.9",
    "langchain-core==1.4.7",
    "langchain-community==0.4.2",
    "langchain-huggingface==1.2.2",
    "langchain-chroma==1.1.0",
    "langchain-text-splitters==1.1.2",
    "sentence-transformers==3.0.1",
    "chromadb",
    "pymupdf",
    "pdfplumber",
    "rank-bm25",
    "bitsandbytes",
    "accelerate",
    "scikit-learn",
    "bert-score",
]

subprocess.run(["pip", "install", "-q", "--no-cache-dir"] + packages, check=True)

#Auto-restart kernel so all installs are visible immediately

#print("✅ Packages installed — restarting kernel...")

#import IPython

#IPython.Application.instance().kernel.do_shutdown(restart=True)

CompletedProcess(args=['pip', 'install', '-q', '--no-cache-dir', 'langchain==1.3.9', 'langchain-core==1.4.7', 'langchain-community==0.4.2', 'langchain-huggingface==1.2.2', 'langchain-chroma==1.1.0', 'langchain-text-splitters==1.1.2', 'sentence-transformers==3.0.1', 'chromadb', 'pymupdf', 'pdfplumber', 'rank-bm25', 'bitsandbytes', 'accelerate', 'scikit-learn', 'bert-score'], returncode=0)

In [3]:
from pathlib import Path
from langchain_core.documents import Document
import fitz        
import pdfplumber
import numpy as np  
pdf_dir = "/kaggle/input/datasets/shivammusk/sec-filings/SEC Filings"
pdf_files = list(Path(pdf_dir).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

all_documents = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()

        if len(text) < 30:
            continue

        # Text block
        all_documents.append(Document(
            page_content=text,
            metadata={
                "source": str(pdf_path),
                "file_name": pdf_path.name,
                "element_type": "Text",
                "page_number": page_num + 1,
            }
        ))

        # Table extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                plumber_page = pdf.pages[page_num]
                tables = plumber_page.extract_tables()
                for idx, table in enumerate(tables):
                    if table and len(table) > 1:
                        table_text = "\n".join(
                            [" | ".join(str(cell) if cell is not None else "" for cell in row)
                             for row in table]
                        )
                        all_documents.append(Document(
                            page_content=table_text,
                            metadata={
                                "source": str(pdf_path),
                                "file_name": pdf_path.name,
                                "element_type": "Table",
                                "page_number": page_num + 1,
                                "table_index": idx,
                            }
                        ))
        except Exception:
            continue

    doc.close()

print(f"\n✅ Extraction complete!")
print(f"Total Documents : {len(all_documents)}")
print(f"Text Blocks     : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Text')}")
print(f"Tables          : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Table')}")


Found 5 PDF files

Processing: Oracle.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing: Meta.pdf
Processing: Tesla.pdf
Processing: Nvidia.pdf
Processing: Apple.pdf

✅ Extraction complete!
Total Documents : 1043
Text Blocks     : 755
Tables          : 288


In [4]:
import yaml

def to_okf_concept(doc):
    company_guess = doc.metadata.get("file_name", "Unknown").split(".")[0].replace("_", " ")
    frontmatter = {
        "type": "FinancialTable" if doc.metadata.get("element_type") == "Table" else "FinancialText",
        "company": company_guess,
        "source_file": doc.metadata.get("file_name", "Unknown"),
        "page": doc.metadata.get("page_number", "?"),
    }
    fm_str = yaml.dump(frontmatter, sort_keys=False)
    doc.page_content = f"---\n{fm_str}---\n\n{doc.page_content.strip()}"
    return doc

all_documents = [to_okf_concept(d) for d in all_documents]
print(f"✅ Wrapped {len(all_documents)} documents as OKF concept blocks (frontmatter + content)")


✅ Wrapped 1043 documents as OKF concept blocks (frontmatter + content)


In [5]:
import subprocess
import sys

# 1. Wipe ALL related cached modules
to_remove = [k for k in sys.modules if any(x in k for x in 
    ["sentence", "langchain_huggingface", "huggingface", "langchain_core", "langchain"])]
for mod in to_remove:
    del sys.modules[mod]

# 2. Reinstall both together
subprocess.run(["pip", "install", "-q", "--no-cache-dir",
    "sentence-transformers==3.0.1",
    "langchain-huggingface==1.2.2"], check=True)

# 3. Verify sentence_transformers loads directly first
import sentence_transformers
print("sentence_transformers version:", sentence_transformers.__version__)

# 4. Now load HuggingFaceEmbeddings fresh
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-08-09 05:45:20.847242: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786254321.074963     147 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786254321.140590     147 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786254321.679716     147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

sentence_transformers version: 3.0.1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [6]:
pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 1.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
# CELL 6: Faster Semantic Chunking (GPU Optimized)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Use GPU with small batch size
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 2          # Small batch = less memory
    }
)

table_docs = [doc for doc in all_documents if doc.metadata.get("element_type") == "Table"]
text_docs = [doc for doc in all_documents if doc.metadata.get("element_type") == "Text"]

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,    
)

print("🔄 Performing semantic chunking on GPU...")
text_chunks = semantic_splitter.split_documents(text_docs)

chunks = table_docs + text_chunks

# Cleanup
del embeddings, semantic_splitter
torch.cuda.empty_cache()
gc.collect()

print(f"✅ Done!")
print(f"Tables: {len(table_docs)} | Text Chunks: {len(text_chunks)} | Total: {len(chunks)}")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

/tmp/ipykernel_147/160383551.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Current GPU memory: 0.41 GB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Performing semantic chunking on GPU...
✅ Done!
Tables: 288 | Text Chunks: 1848 | Total: 2136
GPU memory now: 0.42 GB


In [8]:
sample_embedding = embedding_model.embed_query(
    chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

Embedding dimension: 768


In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./financial_db" 
)

print("✅ Vector Store Created and Persisted")

✅ Vector Store Created and Persisted


In [10]:
query = "What is NVIDIA's total revenue?"
results = vectorstore.similarity_search(query,k = 3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata['file_name']} | Page: {doc.metadata.get('page_number')}")
    print(f"Type: {doc.metadata['element_type']}")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)


--- Result 1 ---
Source: Nvidia.pdf | Page: 94
Type: Text
---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 94
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Notes to the Consolidated Financial Statements
(Continued)
We recognized revenue of $974 million and $729 million in fiscal years 2026 and 2025, respectively, that were included in the prior year
end deferred revenue balance. As of January 25, 2026, revenue related to remaining performance obligations from contracts greater than one year in length was $2.3
billion, ...

--- Result 2 ---
Source: Nvidia.pdf | Page: 57
Type: Table
---
type: FinancialTable
company: Nvidia
source_file: Nvidia.pdf
page: 57
---

Revenue | 100.0 | % |  | 100.0 | %
Cost of revenue | 28.9 |  |  | 25.0 | 
Gross profit | 71.1 |  |  | 75.0 | 
Operating expenses |  |  |  |  | 
Research and development | 8.6 |  |  | 9.9 | 
Sales, general and administrative | 2.1 |  |  | 2.7 | 
Total operating expenses | 10.7 |  |  | 12.6 | 
Opera

In [11]:
!pip install -q bitsandbytes accelerate

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1700,
    temperature=0.4,        
    top_p=0.92,
    do_sample=True,
    repetition_penalty=1.15,
    return_full_text=False,
)

# Plain LLM (used inside financial_rag() for the two-pass generation)
llm = HuggingFacePipeline(pipeline=pipe)

# Chat-formatted wrapper (used by the tool-calling agent below)
chat_llm = ChatHuggingFace(llm=llm)

print("mistralai/Mistral-7B-Instruct-v0.3 (4-bit).")
print("`llm`      -> plain text-completion, used by financial_rag()")
print("`chat_llm` -> chat + tool-calling wrapper, used by the agent")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


Llama-3.1-8B-Instruct loaded (4-bit).
`llm`      -> plain text-completion, used by financial_rag()
`chat_llm` -> chat + tool-calling wrapper, used by the agent


In [13]:
!pip install -q langchain langchain-community

In [14]:
import os
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── ChatMessageHistory: moved to langchain-core in 1.x 
from langchain_core.chat_history import InMemoryChatMessageHistory
from types import SimpleNamespace

# Cross Encoder for reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Per-session memory store
_memory_store = {}

def get_memory(company=None, session_id="default"):
    key = (company.lower().strip() if company else None, session_id)
    if key not in _memory_store:
        _memory_store[key] = InMemoryChatMessageHistory()
    return _memory_store[key]

def _strip_sec_header(text: str) -> str:
    if "[SEC FILING DATA]" not in text:
        return text.strip()
    parts = text.split("---\n", maxsplit=2)
    return parts[2].strip() if len(parts) >= 3 else text.strip()

def _clean_text(text):
    markers = ["<think>", "</think>", "**Final Answer**", "Final Answer:", "Changes made:"]
    for m in markers:
        if m in text:
            text = text.split(m)[0]
    return re.sub(r'\n+(I have|Note that|Please note).*', '', text,
                  flags=re.IGNORECASE | re.DOTALL).strip()

print("✅ Core utilities loaded")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Core utilities loaded


In [15]:
# Hybrid Retrieval (Semantic + BM25)
def hybrid_retrieval(query, vectorstore, company=None, k=50):
    # 1. Semantic search
    results = vectorstore.similarity_search_with_score(query, k=k * 3 if company else k)

    semantic_list = []
    for doc, score in results:
        if company and company.lower() not in doc.metadata.get("source", "").lower():
            continue
        semantic_list.append((doc, 1.0 / (1.0 + score)))

    # 2. BM25 keyword search
    all_data = vectorstore._collection.get(include=["documents", "metadatas"])
    filtered_texts, filtered_metas = [], []

    for text, meta in zip(all_data["documents"], all_data["metadatas"]):
        if company and company.lower() not in str(meta.get("source", "")).lower():
            continue
        filtered_texts.append(_strip_sec_header(text))
        filtered_metas.append(meta)

    bm25_list = []
    if filtered_texts:
        bm25 = BM25Okapi([t.lower().split() for t in filtered_texts])
        scores = bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        for i in top_idx:
            if scores[i] > 0:
                score_norm = scores[i] / max(scores.max(), 1)
                doc = SimpleNamespace(page_content=filtered_texts[i], metadata=filtered_metas[i])
                bm25_list.append((doc, score_norm))

    # Merge semantic + BM25
    merged = {id(d[0]): d for d in semantic_list}
    for doc, score in bm25_list:
        merged[id(doc)] = (doc, merged.get(id(doc), (None, 0))[1] + score * 0.7)

    return sorted(merged.values(), key=lambda x: x[1], reverse=True)[:k]


In [16]:
#  Reranking, multimodal boost, corrective RAG
def rerank_with_cross_encoder(query, candidates, top_n=15):
    if not candidates:
        return []
    docs  = [pair[0] for pair in candidates]
    texts = [_strip_sec_header(d.page_content) for d in docs]
    scores = cross_encoder.predict([[query, t] for t in texts])
    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]


def multimodal_boost(reranked_pairs):
    boosted = []
    for doc, score in reranked_pairs:
        new_score = float(score)
        et = doc.metadata.get("element_type", "")
        text = doc.page_content.lower()

        # Strong boost for real tables
        if et == "Table":
            new_score += 0.40

        # Extra boost for income-statement / segment pages
        keywords = [
            "year ended", "consolidated statements of income",
            "revenue by", "$ in millions", "gross profit",
            "operating income","operating expense", "net income"
        ]
        
        if any(k in text for k in keywords):
            new_score += 0.25

        boosted.append((doc, new_score))
    return sorted(boosted, key=lambda x: x[1], reverse=True)


def evaluate_retrieval_quality(query, docs):
    if not docs or len(docs) < 3:
        return False
    combined_text = " ".join(
        _strip_sec_header(doc.page_content)[:1000] for doc, _ in docs[:4]
    ).lower()
    query_words = [w for w in query.lower().split() if len(w) > 3]
    if not query_words:
        return True
    overlap = sum(1 for w in query_words if w in combined_text)
    required_overlap = max(2, len(query_words) // 3)
    print(f"   Retrieval Quality: {overlap}/{required_overlap} words matched")
    return overlap >= required_overlap

print("✅ Reranking + CRAG utilities loaded")


✅ Reranking + CRAG utilities loaded


In [17]:
# Conversation memory helpers
def _build_history_text(memory, max_turns=3):
    msgs  = memory.messages
    pairs = []
    i = 0
    while i < len(msgs) - 1:
        if msgs[i].type == "human" and msgs[i + 1].type == "ai":
            pairs.append((msgs[i].content, msgs[i + 1].content))
            i += 2
        else:
            i += 1
    recent = pairs[-max_turns:]
    if not recent:
        return ""
    lines = ["Previous conversation:"]
    for turn_idx, (q, a) in enumerate(reversed(recent), 1):
        short_a = a[:500] + "…" if len(a) > 500 else a
        lines.append(f"\n[Turn {turn_idx}] User: {q}")
        lines.append(f"AI: {short_a}")
    return "\n".join(lines)

print("✅ Memory helpers loaded")


✅ Memory helpers loaded


In [18]:
# Main financial_rag() function 
def financial_rag(query: str, company: str = None, session_id: str = "default"):
    global vectorstore, embedding_model, llm

    if not all([vectorstore, embedding_model, llm]):
        return "❌ Error: vectorstore, embedding_model or llm not initialized."

    memory = get_memory(company, session_id)

    # Truncate query for clean logging (no prompt leak)
    query_preview = (query.strip()[:60] + "...") if len(query.strip()) > 60 else query.strip()
    print(f"🔍 Company: {company or 'All'} | Query: {query_preview}")

    # ── Retrieval ──────────────────────────────────────────────────────────
    candidates = hybrid_retrieval(query, vectorstore, company=company, k=40)
    reranked   = rerank_with_cross_encoder(query, candidates, top_n=7)
    reranked   = multimodal_boost(reranked)

    # ── Corrective RAG ─────────────────────────────────────────────────────
    if not evaluate_retrieval_quality(query, reranked):
        print("⚠️  Corrective RAG triggered — widening search...")
        candidates = hybrid_retrieval(query, vectorstore, company=company, k=50)
        reranked   = rerank_with_cross_encoder(query, candidates, top_n=8)
        reranked   = multimodal_boost(reranked)

    # ── Filter & cap ───────────────────────────────────────────────────────
    filtered_docs = [
        (doc, score) for doc, score in reranked
        if not company or company.lower() in str(doc.metadata.get("source", "")).lower()
    ][:16]

    if len(filtered_docs) < 3:
        return f"❌ Not enough relevant information found for '{company}'."

    context = "\n\n---\n\n".join(_strip_sec_header(doc.page_content) for doc, _ in filtered_docs)

    # ── Pass 1: Generate Response ───────────────────────────────────────────
    print("📝 Pass 1: Generating Response")
    pass1_prompt = f"""You are a Senior Institutional Financial Analyst.

ABSOLUTE RULES — NEVER BREAK THEM:

1. Use ONLY numbers that appear VERBATIM in the Context below.
2. Use the context provided below
3. Never invent, estimate, round, or pull any number from memory or training data.
4. LABEL LOCKING (critical):
   - Every number must stay attached to the exact same label it has in the Context.
   - Operating Expenses numbers can ONLY be used for Operating Expenses.
   - Operating Income numbers can ONLY be used for Operating Income.
   - Revenue numbers can ONLY be used for Revenue.
   - Gross Profit / Gross Margin numbers can ONLY be used for Gross Profit / Gross Margin.
   - Net Income numbers can ONLY be used for Net Income.
   - Research & Development, SG&A, and other line items must also keep their own numbers.
   - Never swap or mix numbers between different metrics.

5. When you write a number, always pair it with its full correct label, for example:
   “Operating expenses were $23,076 million”
   “Operating income was $130,387 million”
   Never write a bare number and later assign it to a different metric.

6. DIRECTION RULE:
   - Before writing “increased”, “decreased”, “rose”, “declined”, “growth”, or “drop”, 
     first compare the two numbers belonging to the SAME metric.
   - Later > Earlier → must say “increased” or “rose”.
   - Later < Earlier → must say “decreased” or “declined”.
   - Never assume direction from the movement of expenses or from surrounding text.

7. If either year is missing for a metric, write exactly: “percentage change not available in the retrieved sections.”

9. If a metric is not present in the Context, write: “not disclosed in the retrieved sections.”

11. Write in clear Finincail tone

FIRST, decide the type of question:

A. If the question is mainly about financial figures, trends, revenue, margins, expenses, income, cash flow, etc.:
   → Present the key figures in a clean Markdown table with columns such as:
     | Metric                  | Earlier Year | Later Year | Change | % Change          |
     |-------------------------|--------------|------------|--------|-------------------|
   → After the table, write a structured analysis using only the numbers from the table.
   → For every % Change, show the calculation (example: ((130387-81453)/81453 × 100 = 60.1%)).

B. If the question is about architecture, technology, products (Blackwell, Rubin, etc.), strategy, risks, competition, outlook, or any non-numeric topic:
   → Do NOT create a financial table.
   → Directly write a clear, structured analysis based on the Context.
   → Only mention numbers if they are relevant and present in the Context.

**Context:**
{context}

**Question:**
{query}

**Output Format:**
Provide a structured, concise, and accurate financial analysis.

Financial Analysis:"""

    raw_pass1 = llm.invoke(pass1_prompt)
    final_response = raw_pass1.content if hasattr(raw_pass1, "content") else str(raw_pass1)
    final_response = _clean_text(final_response)


   
    # ── Memory ─────────────────────────────────────────────────────────────
    memory.add_user_message(query)
    memory.add_ai_message(final_response)

    # ── Sources ────────────────────────────────────────────────────────────
    sources = [
        f"{doc.metadata.get('file_name', 'Unknown')} | Page {doc.metadata.get('page_number', '?')}"
        for doc, _ in filtered_docs
    ]
    unique_sources = list(dict.fromkeys(sources))

    final_output = (
        f"# Financial Analysis — {company or 'All Companies'}\n\n"
        + final_response
        + "\n\n## Sources\n"
        + "\n".join(f"- {s}" for s in unique_sources)
    )

    # Debug metadata
    financial_rag._last_context = context
    financial_rag._last_sources = unique_sources
    financial_rag._debug = {
        "initial_docs": len(candidates),
        "final_docs": len(filtered_docs),
        "corrective_triggered": not evaluate_retrieval_quality(query, reranked),
    }

    return final_output

print("✅ financial_rag() with refinement pass ready")


✅ financial_rag() with refinement pass ready


In [19]:
# CELL 17b: BERTScore — measures how well the generated answer is grounded in retrieved context 
from bert_score import score as bert_score

_bert_log = []  # stores {query, company, precision, recall, f1} for every call

def compute_bertscore(reference_text: str, candidate_text: str):
    """
    BERTScore of candidate_text against reference_text.
    Here reference = retrieved source context, candidate = generated answer.

    Unlike BLEU, this compares contextual embeddings of tokens instead of
    exact word matches, so a well-paraphrased but accurate answer still
    scores high. Used here as a grounding/faithfulness proxy:
      - High F1  -> answer's meaning is well supported by the retrieved context
      - Low F1   -> answer may be drifting from / hallucinating beyond the context

    Returns (precision, recall, f1) as plain floats.
    """
    if not reference_text.strip() or not candidate_text.strip():
        return 0.0, 0.0, 0.0

    # BERTScore compares sentence-by-sentence internally; long inputs are fine,
    # but we truncate extremely long context purely to keep this fast.
    ref = reference_text[:4000]
    cand = candidate_text[:4000]

    P, R, F1 = bert_score(
        [cand], [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
    )
    return P.item(), R.item(), F1.item()


def financial_rag_with_bertscore(query: str, company: str = None, session_id: str = "default"):
    """
    Thin wrapper around financial_rag() that additionally computes and prints
    a BERTScore for the generated response (answer vs. retrieved context),
    and logs it to _bert_log for later inspection / averaging.
    """
    response = financial_rag(query, company=company, session_id=session_id)

    context = getattr(financial_rag, "_last_context", "")
    precision, recall, f1 = compute_bertscore(context, response) if context else (0.0, 0.0, 0.0)

    print(f"📊 BERTScore (answer vs. retrieved context) — Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    _bert_log.append({
        "query": query.strip()[:80],
        "company": company or "All",
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    })

    return response


# Backward-compatible alias, in case earlier cells still call the old name
financial_rag_with_bleu = financial_rag_with_bertscore


def show_bertscore_log():
    """Pretty-print the BERTScore for every query run so far."""
    if not _bert_log:
        print("No queries logged yet.")
        return
    print(f"{'#':<3} {'Company':<10} {'Precision':<10} {'Recall':<10} {'F1':<8} Query")
    print("-" * 100)
    for i, entry in enumerate(_bert_log, 1):
        print(f"{i:<3} {entry['company']:<10} {entry['precision']:<10} {entry['recall']:<10} {entry['f1']:<8} {entry['query']}")
    avg_p = sum(e['precision'] for e in _bert_log) / len(_bert_log)
    avg_r = sum(e['recall'] for e in _bert_log) / len(_bert_log)
    avg_f1 = sum(e['f1'] for e in _bert_log) / len(_bert_log)
    print("-" * 100)
    print(f"Average -> Precision: {avg_p:.4f} | Recall: {avg_r:.4f} | F1: {avg_f1:.4f}")

print("✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores")
print("✅ Call show_bertscore_log() anytime to see all scores so far")


✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores
✅ Call show_bertscore_log() anytime to see all scores so far


In [20]:
from IPython.display import display, Markdown

query = """Provide a comprehensive financial analysis of NVIDIA using the latest SEC 10-K filing.

Focus on:
- Total revenue breakdown and year-over-year growth (Data Center vs Gaming vs Professional Visualization vs Automotive)
- Data Center segment performance, including AI infrastructure demand drivers
- Discuss What are the Gross margin trends and key factors affecting profitability
- Discuss what are the Operating expenses, operating income, and net income trends
- Cash flow generation, capital expenditures, and liquidity position
- Key financial highlights and management commentary on future outlook


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""



response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a comprehensive financial analysis of NVIDIA using t...
   Retrieval Quality: 20/25 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 20/25 words matched


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

📊 BERTScore (answer vs. retrieved context) — Precision: 0.7526 | Recall: 0.7626 | F1: 0.7576


# Financial Analysis — Nvidia

NVIDIA Corporation (NVDA)
==============================================

### Total Revenue Breakdown and Year-Over-Year Growth

| Metric               | Earlier Year | Later Year | Change | % Change          |
|----------------------|--------------|------------|--------|-------------------|
| Data Center          | $115,186      | $193,737    | $78,551 | 67.9%              |
| Gaming              | $11,350       | $16,042     | $4,692  | 41.4%              |
| Professional Viz.    | $1,878        | $3,191      | $1,313  | 69.3%              |
| Automotive           | $1,694        | $2,349      | $655    | 38.6%              |
| OEM and Others      | $389          | $619        | $230    | 58.2%              |
| **Total Revenue**   | $130,497     | $215,938    | $85,441 | 65.4%              |

NVIDIA's total revenue experienced a substantial 65.4% year-over-year growth, reaching $215.9 billion in FY 2026. This impressive expansion was primarily driven by the Data Center segment, which accounted for 90% of the overall revenue growth ($78.6 billion). The Gaming segment followed closely behind with a 41.4% YoY growth, while the Professional Visualization and Automotive segments saw respective increases of 69.3% and 38.6%.

### Data Center Segment Performance

The Data Center segment represents the core of NVIDIA's business growth, accounting for approximately 90% of the overall revenue growth in FY 2026. The surge in demand for the Blackwell computing platform drove a 59% YoY increase in Data Center revenue. This growth underscores the importance of platform shifts towards accelerated computing and AI solutions. However, the success of the Data Center segment relies heavily on the availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by customers and partners. Any potential resource shortages could negatively impact future revenue and financial performance.

### Gross Margin Trends and Key Factors Affecting Profitability

| Metric            | Earlier Year | Later Year | Change | % Change          |
|-------------------|--------------|------------|--------|-------------------|
| Gross Profit      | $97,858       | $153,463    | $55,605 | 56.5%              |
| Gross Margin     | 75.0%         | 71.1%       | --      | --                |

NVIDIA's gross profit rose by 56.5% YoY, increasing from $97.9 billion to $153.5 billion in FY 2026. Despite this growth, the gross margin decreased slightly from 75.0% to 71.1%, indicating a shift towards more cost-intensive product lines or lower pricing strategies. It is essential to monitor this trend closely to assess the long-term profitability of the company.

### Operating Expenses, Operating Income, and Net Income Trends

| Metric             | Earlier Year | Later Year | Change | % Change          |
|--------------------|--------------|------------|--------|-------------------|
| Total Operating Expenses | $16,405      | $23,076    | $6,671  | 40.1%              |
| Operating Income     | $81,453      | $130,387    | $48,934 | 60.1%              |
| Net Income          | $72,880      | $120,067    | $47,187 | 65.4%              |

NVIDIA's operating expenses increased by 40.1% YoY, rising from $16.4 billion to $23.1 billion in FY 2026. This rise was primarily driven by research and development expenses, which grew by 24.7% YoY, and sales, general, and administrative expenses, which increased by 16.8% YoY. Despite the escalation in costs, the company managed to achieve a 60.1% YoY increase in operating income, reaching $130.4 billion in FY 2026. Consequently, net income surged by 65.4% YoY, reaching $120.1 billion.

### Cash Flow Generation, Capital Expenditures, and Liquidity Position

| Metric             | Earlier Year | Later Year | Change | % Change          |
|--------------------|--------------|------------|--------|-------------------|
| Net Cash Provided by Operating Activities | $64,089      | $102,718    | $38,629 | 60.6%              |
| Net Cash Used in Investing Activities    | $20,421      | $52,228    | $31,807 | 153.7%             |
| Net Cash Used in Financing Activities    | $42,359      | $48,474    | $6,115  | 14.4%              |
| Cash and Equivalents                      | $28,511      | $62,586    | $34,075 | 119.7%             |
| Marketable Securities                     | $35,567      | $34,379    | $(1,188) | (-3.3%)             |
| Total Cash, Cash Equivalents, and Marketables | $64,080      | $96,965    | $32,885 | 51.8%              |

NVIDIA demonstrated strong cash flow generation capabilities, with net cash provided by operating activities growing by 60.6% YoY, reaching $102.7 billion in FY 2026. This improvement was primarily driven by higher revenue. On the other hand, cash used in investing activities increased substantially by 153.7% YoY, largely due to higher purchases of equity investment securities and the execution of a non-exclusive license agreement with Groq. Cash used in financing activities also increased by 14.4% YoY, mostly due to higher share repurchases. As of January 25, 2026, NVIDIA held $62.6 billion in cash, cash equivalents, and marketable securities, providing ample liquidity to fund future capital requirements.

### Key Financial Highlights and Management Commentary on Future Outlook

NVIDIA reported a robust financial performance in FY 2026, with total revenue growing by 65.4% YoY,

## Sources
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 106
- Nvidia.pdf | Page 72
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 60
- Nvidia.pdf | Page 53

In [21]:
print(financial_rag._last_context)

---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 71
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026
Jan 26, 2025
Jan 28, 2024
Revenue
$
215,938 
$
130,497 
$
60,922 
Cost of revenue
62,475 
32,639 
16,621 
Gross profit
153,463 
97,858 
44,301 
Operating expenses
Research and development
18,497 
12,914 
8,675 
Sales, general and administrative
4,579 
3,491 
2,654 
Total operating expenses
23,076 
16,405 
11,329 
Operating income
130,387 
81,453 
32,972 
Interest income
2,300 
1,786 
866 
Interest expense
(259)
(247)
(257)
Other income, net
9,022 
1,034 
237 
Total other income, net
11,063 
2,573 
846 
Income before income tax
141,450 
84,026 
33,818 
Income tax expense
21,383 
11,146 
4,058 
Net income
$
120,067 
$
72,880 
$
29,760 
Net income per share:
Basic
$
4.93 
$
2.97 
$
1.21 
Diluted
$
4.90 
$
2.94 
$
1.19 
Weighted average shares used in per share compu

In [22]:
query = "Discuss about the architecture of Blackwell and Rubin and how they are benefits to Nvidia in depth "

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about the architecture of Blackwell and Rubin and ho...
   Retrieval Quality: 1/3 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 1/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7982 | Recall: 0.8289 | F1: 0.8133


# Financial Analysis — Nvidia

| Metric                  | Fiscal Year 2025 | Fiscal Year 2026 | Change | % Change         |
|-------------------------|------------------|------------------|--------|------------------|
| Investment in Research & Development | $76,700 million | Not Disclosed    | Increase | Not Available    |
| Launch of Blackwell Architecture | N/A               | Yes              | N/A      | N/A             |
| Launch of Rubin Platform | N/A               | Expected         | N/A       | N/A            |

The NVIDIA Blackwell architecture, launched in fiscal year 2025, represents a significant leap forward in data center scale infrastructure. It includes GPUs, CPUs, DPUs, interconnects, switch chips and systems, and networking adapters, all designed to excel at processing cutting edge generative AI and accelerated computing workloads with market leading performance and efficiency. The architecture offers multiple configurations, catering to various industries and a wide array of AI and accelerated computing use cases.

In fiscal year 2026, NVIDIA unveiled the Rubin platform, which is scheduled to begin production shipments in the second half of the fiscal year. Designed for agentic AI and reasoning, Rubin promises to process multi-step problem-solving and massive long-context workflows, delivering up to a 10x reduction in cost per token compared to Blackwell. Although detailed financial information regarding Rubin's launch is not disclosed, the potential cost savings it brings could significantly contribute to NVIDIA's overall profitability.

By investing heavily in research and development ($76.7 billion since inception), NVIDIA has been able to consistently introduce innovative architectures like Blackwell and Rubin. This continuous investment in innovation allows NVIDIA to maintain technological superiority in its target markets, which include Data Center, Gaming, Professional Visualization, and Automotive. The programmable nature of their architecture enables them to address diverse computing requirements across these markets with a unified underlying architecture, making leveraged investments in research and development possible. Furthermore, the company utilizes this platform approach in each of its target markets, extending its technology and platform leadership in AI.

## Sources
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 9

In [23]:

# CELL 26: NVIDIA — Export Control Risks

from IPython.display import display, Markdown

query = """Provide a detailed analysis of NVIDIA's export control risks, geopolitical exposure, and China-related challenges.

Focus on:
- Impact of U.S. export restrictions on products
- Licensing requirements, revenue impact, and inventory charges
- Competitive effects on China data center market discuss this in detail
- Mitigation strategies and long-term implications
- Broader supply chain and regulatory risks discuss this in detail

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Quote relevant sections from the Risk Factors and Business sections."""

# Run the analysis
response = financial_rag_with_bleu(query, company="Nvidia")

# Display the response
display(Markdown(response))


🔍 Company: Nvidia | Query: Provide a detailed analysis of NVIDIA's export control risks...
   Retrieval Quality: 20/23 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 20/23 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7677 | Recall: 0.7894 | F1: 0.7784


# Financial Analysis — Nvidia

NVIDIA Export Control Risks, Geopolitical Exposure, and China Challenges
==========================================================================================

Export Control Risks
---------------------

### Impact of U.S. export restrictions on products

NVIDIA faces significant challenges due to U.S. export restrictions on products, particularly those associated with Artificial Intelligence (AI). These restrictions target specific parameters like total processing performance, "performance density," interconnect bandwidth, and memory bandwidth of chips [NVIDIA PDF, p. 41]. As of fiscal year 2026, these restrictions have effectively barred NVIDIA from competing in China's data center computing/compute market [NVIDIA PDF, p. 41]. This exclusion has enabled competitors to establish larger developer and customer ecosystems, potentially challenging NVIDIA globally.

### Licensing requirements, revenue impact, and inventory charges

Licensing requirements for selling products in the China market can be temporary, burdensome, or include financial or economic requirements that may not be feasible for NVIDIA or its customers [NVIDIA PDF, p. 15]. These requirements have already and may in the future benefit competitors, making pre-sale and post-sale technical support efforts more cumbersome and less certain [NVIDIA PDF, p. 15]. Moreover, the licensing process may not be resolved before significant business opportunities evaporate [NVIDIA PDF, p. 15]. If approved, licenses may be temporary, imposing burdensome conditions or financial requirements that could negatively impact revenues [NVIDIA PDF, p. 15].

Additionally, restrictions imposed by the Chinese government on the duration of gaming activities and access to games may adversely affect Gaming revenue [NVIDIA PDF, p. 15]. Increased oversight of digital platform companies may also adversely affect Data Center revenue [NVIDIA PDF, p. 15].

### Competitive effects on China data center market

The Chinese government encourages customers to purchase from domestic competitors and discourages them from buying, importing, or using NVIDIA data center products [NVIDIA PDF, p. 41]. This situation creates a competitive disadvantage for NVIDIA in the China data center market. As of fiscal year 2026, NVIDIA was effectively excluded from this market, helping competitors build larger developer and customer ecosystems to challenge NVIDIA worldwide [NVIDIA PDF, p. 41].

## Mitigation Strategies and Long-Term Implications

To mitigate these challenges, NVIDIA aims to enhance the resiliency and redundancy of its supply chain, which is currently concentrated in Asia [NVIDIA PDF, p. 15]. However, new and existing export controls or changes to existing export controls could limit alternative manufacturing locations, potentially impacting the company's business.

Regarding the China market, NVIDIA plans to return with a product that meets the approval of both the U.S. Government and the Chinese government [NVIDIA PDF, p. 41]. However, achieving this goal remains uncertain, given the complexity and geopolitical landscape involved.

## Broader Supply Chain and Regulatory Risks

NVIDIA faces broader supply chain risks due to increased global focus on AI technologies, leading to regulatory restrictions that target products and services capable of enabling or facilitating AI [NVIDIA PDF, p. 37]. These restrictions may impact some or all of NVIDIA's product and service offerings, limiting the ability of downstream customers and users worldwide to acquire, deploy, and use systems that include NVIDIA's products, software, and services [NVIDIA PDF, p. 37].

Moreover, geopolitical tensions have made semiconductors associated with AI, including GPUs and related products, the focus of export control restrictions proposed by stakeholders in the U.S. and its allies [NVIDIA PDF, p. 37]. These controls may further limit NVIDIA's ability to export technology, products, or services, creating a competitive disadvantage and negatively impacting its business, revenue, and financial results.

## Advice for Investors

Investors should closely monitor NVIDIA's progress in addressing export control risks, geopolitical exposure, and China-related challenges. They should consider the potential impact of these issues on the company's financial performance and competitive position. Investors might also want to evaluate NVIDIA's strategic initiatives aimed at enhancing its supply chain resiliency and diversifying its manufacturing locations.

In conclusion, NVIDIA faces significant challenges due to U.S. export restrictions, licensing requirements, competitive pressures in the China data center market, and broader supply chain and regulatory risks associated with AI technologies. To mitigate these challenges, NVIDIA aims to enhance its supply chain resiliency and return with a product that meets the approval of both the U.S. Government and the Chinese government. Investors should carefully assess the potential impact of these issues on NVIDIA's financial performance and competitive position.

## Sources
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 15
- Nvidia.pdf | Page 41
- Nvidia.pdf | Page 37

In [24]:
query = """Evaluate NVIDIA's competitive positioning across its markets.

Focus on:
- Competition in Data Center (AMD, Intel, custom ASICs from hyperscalers)
- Discuss about Gaming GPU competition 
- Professional Visualization and Automotive segments
- Overall technology leadership in GPUs, CUDA, networking, and software
- Barriers to entry and ecosystem strength

Discuss strengths, weaknesses, and investor implications with references from the filing."""

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's competitive positioning across its markets...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 19/14 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 19/14 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7656 | Recall: 0.7553 | F1: 0.7604


# Financial Analysis — Nvidia

Evaluation of NVIDIA's Competitive Positioning Across Its Markets
=====================================================================================

In this analysis, we evaluate NVIDIA's competitive positioning across its primary markets, focusing on Data Center, Gaming, Professional Visualization, and Automotive segments. We discuss the main competitors in each segment, highlighting NVIDIA's unique selling points, barriers to entry, and overall technology leadership.

Data Center Market
------------------

NVIDIA competes in the data center market with established players like AMD and Intel, as well as custom ASICs developed by hyperscalers such as Google, Facebook, and Microsoft. NVIDIA's advantage lies in its Blackwell architectures, which represent the majority of its Data Center revenue [context]. However, the availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by its customers and partners is crucial, and any shortage of these resources could impact NVIDIA's future revenue and financial performance [context].

Gaming GPU Market
------------------

In the highly competitive Gaming GPU market, NVIDIA faces stiff competition from AMD, Intel, and other manufacturers offering integrated GPUs. NVIDIA's strength lies in its superior performance, innovative features, and strong brand recognition. However, export controls have been a significant challenge for NVIDIA in the gaming GPU market, potentially limiting its sales to markets outside China, including the U.S. and Europe [context]. This creates an opening for competitors like AMD to gain market share in these regions.

Professional Visualization and Automotive Segments
----------------------------------------------------

NVIDIA's professional visualization and automotive segments cater to specific niches where its GPUs excel due to their high computational power and programmability. Key competitors in these areas include AMD, Broadcom, Intel, and other suppliers of SoC products. NVIDIA's advantage lies in its full-stack innovation approach, addressing diverse computing requirements across multiple end markets with a unified underlying architecture [context]. This enables NVIDIA to make leveraged investments in research and development, supporting several multi-billion-dollar end markets with shared underlying technology [context].

Overall Technology Leadership
------------------------------

NVIDIA boasts a strong technology leadership position, providing a complete, end-to-end accelerated computing platform for AI, addressing both training and inferencing. This includes full-stack data center-scale compute and networking solutions across processing units, interconnects, systems, and software [context]. NVIDIA offers a wide range of AI-specific features for its GPUs, further extending its leadership position in the AI space. NVIDIA also provides NVIDIA AI Enterprise, a comprehensive software suite designed to simplify the development and deployment of production-grade, end-to-end generative AI applications [context].

Barriers to Entry and Ecosystem Strength
------------------------------------------

NVIDIA's extensive ecosystem, encompassing hardware, software, and strategic partnerships, acts as a barrier to entry for new competitors. NVIDIA's platform approach in each of its target markets allows it to leverage economies of scale and invest in research and development, making it difficult for smaller competitors to match NVIDIA's offerings [context]. Furthermore, NVIDIA's strong relationships with cloud service providers, such as Amazon Web Services, Microsoft Azure, and Google Cloud Platform, contribute to its competitive edge [context].

Investor Implications
----------------------

Investors should consider NVIDIA's competitive positioning when evaluating the company's long-term prospects. Despite facing challenges in certain markets, such as gaming GPUs due to export controls, NVIDIA maintains a strong competitive position in the data center, professional visualization, and automotive markets thanks to its technology leadership, robust ecosystem, and full-stack innovation approach. Investors should monitor regulatory developments affecting export controls and potential impacts on NVIDIA's business, particularly in the data center and gaming markets.

## Sources
- Nvidia.pdf | Page 12
- Nvidia.pdf | Page 9
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 41
- Nvidia.pdf | Page 15

In [25]:
query = """Analyze NVIDIA's long-term corporate strategy, key risks, and growth outlook.

Focus on:
- Platform strategy (hardware + software + ecosystem)
- Expansion into AI, robotics, autonomous driving, and professional visualization
- Supply chain, manufacturing, and capacity risks
- Human capital, R&D investment, and innovation approach
- Major risks from the Risk Factors section and mitigation efforts

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Provide a balanced view with exact quotes and key takeaways for long-term investors."""


response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Analyze NVIDIA's long-term corporate strategy, key risks, an...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 22/22 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 22/22 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7717 | Recall: 0.7584 | F1: 0.7650


# Financial Analysis — Nvidia

NVIDIA's Long-Term Corporate Strategy, Key Risks, and Growth Outlook
==========================================================================================

### Platform Strategy

NVIDIA's platform strategy revolves around its accelerated computing platform, which combines hardware, software, and an extensive ecosystem. The company aims to address various markets with a unified underlying architecture, utilizing its GPUs, CPUs, CUDA, and networking technologies as foundational building blocks [1]. By adopting a full-stack innovation approach, NVIDIA delivers order-of-magnitude performance improvements compared to traditional methods in targeted markets like Data Center, Gaming, Professional Visualization, and Autonomous Vehicles [2].

### Expansion into AI, Robotics, Autonomous Driving, and Professional Visualization

NVIDIA has successfully extended its technology and platform leadership in AI by offering a complete, end-to-end accelerated computing platform for AI, encompassing data center-scale compute and networking solutions [3]. This includes all three primary processing units in AI servers: GPUs, CPUs, and DPUs. Moreover, NVIDIA provides NVIDIA AI Enterprise, a comprehensive software suite that simplifies the development and deployment of production-grade, end-to-end generative AI applications [4].

In the realm of robotics, NVIDIA offers DRIVE AGX Orin, a powerful AI computer specifically designed for robots, drones, and automated machines [5]. Furthermore, NVIDIA has made strides in autonomous driving by providing a reference sensor set, running an in-vehicle operating system (DRIVE OS), and an open, modular DRIVE software platform for autonomous driving, mapping, and parking services [6]. Lastly, NVIDIA caters to the Professional Visualization market by delivering high-performance graphics cards and software solutions for creating, editing, and rendering 3D content [7].

### Supply Chain, Manufacturing, and Capacity Risks

One of the critical challenges faced by NVIDIA is ensuring the availability of data centers, energy, and capital required to support the buildout of NVIDIA AI infrastructure by customers and partners [8]. As the demand for AI infrastructure grows, securing adequate energy capacity becomes increasingly complex due to regulatory, technical, and construction hurdles [8]. Additionally, access to capital can be restricted for smaller companies pursuing large-scale infrastructure projects [8].

To mitigate these risks, NVIDIA invests heavily in research and development in markets where it has limited operating history but holds great potential [9]. However, these investments might not generate meaningful revenues for several years or even fail to yield returns altogether [9].

### Human Capital, R&D Investment, and Innovation Approach

NVIDIA emphasizes human capital as a crucial aspect of its success. The company focuses on recruiting top talent, fostering collaboration, and investing in cutting-edge research and development [10]. NVIDIA's innovation approach involves advancing its accelerated computing platform to tackle complex problems more efficiently than traditional computational methods [1].

### Major Risks from the Risk Factors Section and Mitigation Efforts

Regulatory scrutiny represents another significant risk for NVIDIA, especially in relation to its position in the AI market [11]. The company faces increasing interest from global regulators, including the European Union, the United States, the United Kingdom, South Korea, Japan, and China [11]. To manage this risk, NVIDIA cooperates with investigations and complies with regulations as they evolve [11].

Another concern relates to cybersecurity vulnerabilities and risks, necessitating increased compliance costs due to changes or heightened enforcement of antitrust legislation, regulation, and administrative rule-making [12]. NVIDIA addresses this issue by implementing robust security measures and staying abreast of regulatory developments.

### Advice for Investors

Long-term investors seeking exposure to emerging technologies like AI, robotics, and autonomous driving may find NVIDIA an attractive investment opportunity given its strong platform strategy, expansions into various markets, and commitment to innovation. However, investors should remain aware of the risks associated with supply chain constraints, regulatory scrutiny, and cybersecurity concerns. Carefully monitoring NVIDIA's progress in managing these risks and maintaining its technological edge is essential for making informed decisions.

## Sources
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 9

In [26]:
query = "explain the supply chain risk of Nvidia in more depth like who are the key suppliers and what they supply and why it is cruical for Nvidia and also tell based on context suggest improvements in depth."

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))


🔍 Company: Nvidia | Query: explain the supply chain risk of Nvidia in more depth like w...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 10/7 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 10/7 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7942 | Recall: 0.7574 | F1: 0.7754


# Financial Analysis — Nvidia

Supply Chain Risks for Nvidia

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| Key Suppliers' Revenue  | Not Disclosed | Not Disclosed | Not Available | -       |

The primary concern lies within the supply chain of Nvidia, specifically with key suppliers whose components are crucial for the production of various Nvidia products, particularly GPUs. These suppliers include manufacturers of silicon wafers, packaging materials, and assembly equipment.

The importance of these suppliers stems from their role in providing essential raw materials and manufacturing services necessary for producing high-quality GPUs that meet Nvidia's stringent specifications. A disruption in the supply chain could lead to delayed product launches, increased costs, and decreased revenue due to missed opportunities.

To mitigate this risk, Nvidia should diversify its supplier base by establishing relationships with alternative suppliers in different geographic locations. This approach will help ensure a steady supply of critical components while reducing dependence on individual suppliers. Additionally, maintaining open communication channels with key suppliers to discuss potential issues early on can aid in proactive problem-solving and contingency planning.

Moreover, focusing on developing an efficient and flexible supply chain management system will enable Nvidia to respond quickly to changes in the market and minimize the impact of potential disruptions. Implementing agile manufacturing techniques, improving forecasting accuracy, and optimizing inventory levels can contribute to a more robust and resilient supply chain.

Lastly, Nvidia should invest in research and development aimed at creating innovative solutions that require fewer resources or alternative materials. By doing so, the company can reduce reliance on specific suppliers and improve overall supply chain efficiency.

## Sources
- Nvidia.pdf | Page 27
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 34
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 24

In [27]:
query = "Discuss about operating expenses of Nvidia in depth"

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))


🔍 Company: Nvidia | Query: Discuss about operating expenses of Nvidia in depth


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 3/2 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 3/2 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7949 | Recall: 0.7988 | F1: 0.7969


# Financial Analysis — Nvidia

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| Total operating expenses | $16,405      | $23,076    | $6,671 | 41%               |
| Research and development | $12,914      | $18,497    | $5,583 | 43%               |
| Sales, general and administrative | $3,491       | $4,579     | $1,088 | 31%               |

Overall, Nvidia's total operating expenses increased by 41%, going from $16,405 million to $23,076 million between the earlier and later years. This rise in expenses can be attributed to an increase in both research and development (R&D) expenses and sales, general, and administrative (SG&A) expenses.

More specifically, R&D expenses rose by 43% from $12,914 million to $18,497 million, while SG&A expenses grew by 31% from $3,491 million to $4,579 million. It should be noted that these percentages represent increases over the absolute values of each respective expense category.

The growth in operating expenses reflects Nvidia's continued investment in innovation and expansion, which may include initiatives like product development, marketing efforts, and hiring additional personnel. However, it remains crucial for investors to monitor this trend closely to assess the company's ability to generate sufficient profits and maintain a healthy balance sheet.

## Sources
- Nvidia.pdf | Page 105
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 53
- Nvidia.pdf | Page 58
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 108

In [28]:
query = """Evaluate NVIDIA's overall corporate strategy and capital allocation decisions.

Focus on:
- Core business model and diversification efforts
- R&D investment trends
- Mergers & acquisitions strategy
- Share buyback and dividend policy
- Long-term vision in AI, robotics, autonomous vehicles, and Omniverse
- Management's capital allocation priorities

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 

Provide investor implications."""

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's overall corporate strategy and capital all...
   Retrieval Quality: 11/18 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 11/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7516 | Recall: 0.7552 | F1: 0.7534


# Financial Analysis — Nvidia

Evaluating NVIDIA's Overall Corporate Strategy and Capital Allocation Decisions
==============================================================================================

### Core Business Model and Diversification Efforts

NVIDIA's core business model revolves around accelerated computing, focusing on various computationally intensive fields like AI, data science, autonomous vehicles, robotics, and digital twins. Originally focused on PC graphics, the company has successfully expanded beyond this niche market, becoming a data center-scale AI infrastructure company transforming multiple industries.

### R&D Investment Trends

Investment in research and development (R&D) plays a crucial role in NVIDIA's long-term success. In fiscal year 2026, the company spent approximately $17.5 billion on private companies and infrastructure funds, primarily supporting early-stage startups. While these investments are illiquid and non-marketable, they contribute to the development of cutting-edge technologies within the AI ecosystem. Furthermore, the company invests in publicly-traded equities whose values may fluctuate significantly due to changes in stock prices, impacting NVIDIA's financial results.

### Mergers & Acquisitions Strategy

NVIDIA's merger and acquisition (M&A) strategy focuses on acquiring companies that complement its existing product lines and technology stack. However, the context does not provide specific details on recent M&As undertaken by the company. It is essential for investors to monitor future M&A activity to assess the strategic fit and financial impact of these transactions.

### Share Buyback and Dividend Policy

NVIDIA does not seem to have a formal share buyback program mentioned in the provided context. Regarding dividends, the company states that it plans to pay regular quarterly dividends in the future; however, no specific amounts or timelines are given. Investors should closely follow updates on NVIDIA's dividend policy to gauge the company's commitment to returning capital to shareholders.

### Long-Term Vision in AI, Robotics, Autonomous Vehicles, and Omniverse

NVIDIA's long-term vision lies in the continued expansion of its AI capabilities across various sectors, including robotics, autonomous vehicles, and Omniverse – a virtual world simulation platform. The company aims to leverage its expertise in accelerated computing to drive innovation in these areas, reshaping industries and creating new opportunities for growth.

### Management's Capital Allocation Priorities

Management prioritizes investing in R&D, strategic partnerships, and infrastructure to support the development of advanced technologies and the broader AI ecosystem. Additionally, NVIDIA makes land, power, and shell guarantees to early-stage companies, often over multi-year periods, to facilitate the construction of complex data center infrastructures. These commitments demonstrate management's dedication to fostering a robust technological foundation for the company's future growth.

### Investor Implications

For investors considering NVIDIA as a potential investment, the following aspects are worth noting:

* **Long-term Growth Opportunities**: NVIDIA's focus on AI, robotics, autonomous vehicles, and Omniverse presents exciting long-term growth prospects, especially as these technologies increasingly permeate various industries.
* **R&D Investments**: NVIDIA's substantial investments in R&D underscore the company's commitment to staying at the forefront of technological advancements, ensuring continued innovation and competitive advantage.
* **Capital Allocation Priorities**: Management's emphasis on strategic partnerships, infrastructure investments, and R&D expenditures suggests a forward-looking approach aimed at driving sustainable growth.
* **Regulatory Risks**: Given the increasing scrutiny from regulators worldwide, particularly in relation to AI and competition concerns, investors should remain vigilant about potential regulatory headwinds that might impact NVIDIA's financial performance.
* **Macroeconomic Factors**: Various macroeconomic factors, such as tariffs, inflation, interest rates, capital market volatility, global supply chain constraints, and geopolitical developments, can have both direct and indirect effects on NVIDIA's results of operations, particularly demand for its products.

In conclusion, NVIDIA's strong focus on AI, robotics, autonomous vehicles, and Omniverse, combined with substantial R&D investments and strategic partnerships, positions the company well for long-term growth. However, investors should carefully consider the potential risks associated with regulatory oversight and macroeconomic factors when deciding whether to invest in NVIDIA.

## Sources
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 81
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 83
- Nvidia.pdf | Page 53

In [29]:
query = """Provide a comprehensive financial analysis of Tesla using the latest SEC 10-k filings.

Focus on:
- Total revenue breakdown in 2025 vs 2024 vs 2023 and year-over-year growth trends (Automotive vs Energy Generation & Storage)
- Discuss Automotive segment performance 
- Discuss Energy Generation and Storage segment growth and margins
- Gross margin trends and key drivers
- Operating income, net income, and profitability trends
- Free cash flow generation and capital expenditure
- Overall financial health and liquidity position


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""


response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Provide a comprehensive financial analysis of Tesla using th...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 37/24 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 37/24 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7471 | Recall: 0.7355 | F1: 0.7412


# Financial Analysis — Tesla

Tesla (FY 2023 – FY 2025)
=============================================

### Total Revenue Breakdown

| Metric               | FY 2025 | FY 2024 | FY 2023 | YoY Growth | % Change |
|---------------------|--------|--------|--------|-----------|----------|
| **Total Revenue**   | $94,827 | $97,690 | $96,773 | 1.3%       | N/A      |
| **Automotive Sales** | $65,821 | $72,480 | $78,509 | -8.8%      | -12.2%    |
| **Automotive Leasing** | $1,712  | $1,827  | $2,120  | -6.6%      | -18.3%    |
| **Energy Generation & Storage** | $12,771 | $10,086 | $6,035  | 26.9%      | N/A      |
| **Services & Other** | $12,530 | $10,534 | $8,319  | 19.4%      | N/A      |

Overall, Tesla experienced a slight increase in total revenue from $96,773 million in FY 2023 to $94,827 million in FY 2025, marking a 1.3% decline year-over-year. This decrease can be attributed to the drop in Automotive Sales (-8.8%) and Automotive Leasing (-6.6%), while Energy Generation & Storage (+26.9%) and Services & Other (+19.4%) segments showed positive growth.

### Automotive Segment Performance

The Automotive segment, comprising Automotive Sales, Automotive Leasing, and Regulatory Credits, saw a combined revenue decline from $80,609 million in FY 2024 to $69,526 million in FY 2025, representing a 12.2% year-over-year reduction. The primary contributors to this decrease were a drop in Automotive Sales (-8.8%) and Automotive Leasing (-6.6%).

#### Automotive Sales

Automotive Sales declined from $72,480 million in FY 2024 to $65,821 million in FY 2025, reflecting a 9.0% year-over-year decrease. This trend suggests potential market saturation or competitive pressures affecting Tesla's ability to maintain previous sales levels.

#### Automotive Leasing

Automotive Leasing followed a similar trajectory, dropping from $1,827 million in FY 2024 to $1,712 million in FY 2025, resulting in a 6.6% year-over-year decrease. Although this figure represents a smaller percentage of overall revenue, it still indicates a declining trend in this area.

### Energy Generation and Storage Segment Growth and Margins

The Energy Generation & Storage segment demonstrated robust growth, increasing from $6,035 million in FY 2023 to $12,771 million in FY 2025, representing a 26.9% year-over-year expansion. This growth can be attributed to factors such as increased deployment of Megapack and Powerwall solutions.

Despite the impressive revenue growth, the gross margin for this segment decreased from 22.5% in FY 2024 to 18.4% in FY 2025. This contraction may be due to the decrease in average selling prices of Megapack, partially offsetting the revenue growth.

### Gross Margin Trends and Key Drivers

Tesla's overall gross margin decreased from 18.4% in FY 2024 to 17.8% in FY 2025, primarily driven by the decline in regulatory credits revenue and changes in both Automotive Sales and Automotive Cost of Sales. The Energy Generation & Storage segment's gross margin also dropped slightly, from 22.5% in FY 2024 to 18.4% in FY 2025, due to the decrease in average selling prices of Megapack.

### Operating Income, Net Income, and Profitability Trends

Tesla's operating income fell significantly from $7,076 million in FY 2024 to $4,355 million in FY 2025, indicating a 40.1% year-over-year decrease. This decline can be linked to the reduction in gross profit across various segments, particularly Automotive Sales and Energy Generation & Storage.

Net income followed a similar pattern, decreasing from $7,091 million in FY 2024 to $3,794 million in FY 2025, representing a 46.5% year-over-year decline. Despite the drop in net income, Tesla remains profitable, demonstrating resilience amidst challenging market conditions.

### Free Cash Flow Generation and Capital Expenditure

Tesla generated $1.14 billion in free cash flow during FY 2025, a substantial improvement from the negative $3.85 billion in FY 2024. This turnaround highlights the company's improved operational efficiency and cash management. However, it should be noted that Tesla invested heavily in capital expenditures ($15.7 billion in FY 2025), suggesting continued focus on expanding production capacity and research & development initiatives.

### Overall Financial Health and Liquidity Position

With $17,616 million in cash and cash equivalents as of December 31, 2025, Tesla maintains a strong liquidity position. Furthermore, the company's total debt stood at $3.05 billion, down from $3.85 billion in FY 2024, indicating a more manageable debt load. These factors suggest that Tesla is financially healthy and capable of navigating short-term challenges while continuing to invest in long-term growth opportunities.

### Investor Advice

For investors seeking exposure to cutting-edge technologies and sustainable energy solutions, Tesla presents an attractive opportunity. Despite facing headwinds in certain areas, the company's Energy Generation & Storage segment shows promise for future growth. Long-term investors may find value in Tesla's innovative approach to AI-driven transportation and robotics, as the company positions itself to lead the transition towards a more automated and sustainable world.

## Sources
- Tesla.pdf | Page 54
- Tesla.pdf | Page 66
- Tesla.pdf | Page 64
- Tesla.pdf | Page 77
- Tesla.pdf | Page 74
- Tesla.pdf | Page 99
- Tesla.pdf | Page 82

In [30]:
print(financial_rag._last_context)

Management’s Discussion and Analysis of Financial Condition and Results
of Operations in our Annual Report on Form 10-K for fiscal year 2024, which was filed with the SEC on January 30, 2025. Overview and 2025 Highlights
We are focused on bringing artificial intelligence into the real world, through products and services like FSD (Supervised) and
Robotaxi, as well as working to develop and commercialize AI robots (including Optimus). We intend to leverage our current
operations, in which we design, develop, manufacture, sell and lease high-performance fully electric vehicles and energy generation
and storage systems that increasingly deliver AI-related and enhanced software and services to our customers, to achieve that objective. As a result of rapidly evolving trade and fiscal policy, uncertainty in the automotive and energy markets continues, posing risks
to our global supply chain and cost structure which could have a meaningfully adverse impact on demand for our products and our
p

In [31]:
query = "Discuss about operating expenses of Tesla in depth"

"Quote exact figures from given retrived context you received and compare correct and accurate trends where available."

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Discuss about operating expenses of Tesla in depth


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 3/2 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 3/2 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7407 | Recall: 0.8207 | F1: 0.7786


# Financial Analysis — Tesla

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| Research and development | $ 6,411      | $ 12,739    | $ 6,328 | 100.5%             |
| Selling, general and administrative | $ 5,150      | $ 5,834     | $ 684   | 13.3%              |
| Restructuring and other | $ 684        | Not disclosed in the retrieved sections | Not applicable | N/A                |
| Total operating expenses | $ 12,739    | Not disclosed in the retrieved sections | Not applicable | N/A                |

From the provided financial statement, we can see the operating expenses of Tesla for the year ending December 31, 2025, and the previous year. The following is a detailed breakdown of each component of operating expenses:

1. **Research and development (R&D)** expenses increased significantly from $6,411 million in 2024 to $12,739 million in 2025, representing a 100.5% rise. This increase is largely attributed to higher costs related to Artificial Intelligence (AI) and other programs as Tesla expands its product roadmap and technological capabilities. Additionally, there was a $500 million increase in stock-based compensation. As a result, R&D expenses accounted for 7% of total revenues in 2025, up from 5% in 2024.

2. **Selling, general and administrative (SG&A)** expenses rose by 13.3% from $5,150 million in 2024 to $5,834 million in 2025. This increase is mainly due to a $354 million increase in operating expenses, including legal charges, a $256 million increase in employee and labor costs, a $235 million increase in stock-based compensation, partially offset by an $83 million decrease in marketing expenses and a $78 million decrease in facilities-related expenses. Consequently, SG&A expenses represented 6% of total revenues in 2025, up from 5% in 2024.

3. **Restructuring and other** expenses are not disclosed in the provided financial statements for the year ending December 31, 2025. Therefore, we cannot determine any change in this category compared to the previous year.

4. **Total operating expenses** are not disclosed in the provided financial statements for the year ending December 31, 2025. Thus, we cannot calculate the overall change in operating expenses compared to the previous year. However, considering the individual components, it is evident that operating expenses have risen substantially due to increased spending on research and development and selling, general, and administrative expenses.

## Sources
- Tesla.pdf | Page 120
- Tesla.pdf | Page 82
- Tesla.pdf | Page 66
- Tesla.pdf | Page 68
- Tesla.pdf | Page 72

In [32]:
query = """Analyze Tesla's autonomous driving technology, Full Self-Driving (FSD), and Robotaxi strategy in detail.

Focus on:
- Current status and capabilities of FSD (Supervised)
- Progress toward unsupervised autonomy and regulatory approvals
- Robotaxi business model and deployment plans
- Competitive advantages in AI and neural networks
- Timeline and risks for commercialization
- Management's vision and expected financial impact



Explain each point above in very much depth and with intuition

Quote specific statements from the filings and conclude with investor implications."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Analyze Tesla's autonomous driving technology, Full Self-Dri...
   Retrieval Quality: 16/19 words matched
⚠️  Corrective RAG triggered — widening search...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 16/19 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7725 | Recall: 0.7902 | F1: 0.7812


# Financial Analysis — Tesla

Tesla’s Autonomous Driving Technology and Robotaxi Strategy
=============================================================================

Tesla’s Autonomous Driving Technology – FSD (Supervised)
------------------------------------------------------

* **Current Status and Capabilities**: Tesla has been focusing on bringing AI into the real world through its proprietary Full Self-Driving (FSD) technology. The company uses the term "FSD (Supervised)" for its autonomous driving system, which includes the equivalent naming convention, "FSD (Capability)", in the European, Middle East, and Asia-Pacific regions. [Source: Page 6]
* **Progress Toward Unsupervised Autonomy and Regulatory Approvals**: Although Tesla launched its Robotaxi service in June 2025, there is no explicit information about the progress towards unsupervised autonomy or regulatory approvals. However, the company mentions that it expects to advance a service-driven business model based on AI, software, and fleet-based profits, indicating ongoing development efforts. [Source: Page 6]

Robotaxi Business Model and Deployment Plans
---------------------------------------------

* **Business Model**: Tesla’s Robotaxi service aims to open access to an expanded customer base as modes of transportation evolve. By providing an autonomous ride-hailing platform, the company hopes to unlock the potential to advance a service-driven business model based on AI, software, and fleet-based profits. Alongside FSD (Supervised) subscriptions, this could lead to increased revenues and profitability. [Source: Page 6]
* **Deployment Plans**: Initially, Tesla’s Robotaxi service operates with Model Y vehicles. Over time, the company plans to introduce Cybercab, a purpose-built autonomous vehicle designed specifically for the service. This expansion indicates a long-term commitment to the Robotaxi business. [Source: Page 6]

Competitive Advantages in AI and Neural Networks
--------------------------------------------------

* **AI Investments**: Tesla continues to invest heavily in AI research and development, aiming to improve its FSD (Supervised) capabilities and neural network algorithms. These investments position the company to remain competitive in the rapidly evolving field of autonomous driving. [Source: Page 6]
* **Scalable Mobility Infrastructure**: By building a robust Supercharger network and enhancing infotainment offerings, Tesla creates a scalable mobility infrastructure that supports its Robotaxi service. This advantage positions the company to cater to a wider range of customers and adapt to changing market demands. [Source: Page 6]

Timeline and Risks for Commercialization
------------------------------------------

* **Commercialization Timeline**: While Tesla launched its Robotaxi service in June 2025, there is no specific timeline provided for when the company expects to see significant commercial success or mass production of Cybercab. [Source: Page 6]
* **Risks**: The success of Tesla’s Robotaxi service depends on several factors, including consumer acceptance of autonomous driving solutions, the preference for Robotaxi over traditional ride-hailing and taxi services, and the development of the autonomous driving market amid growing competition. Additionally, the company faces risks associated with unfavorable global market conditions and operational challenges. [Source: Page 6]

Management’s Vision and Expected Financial Impact
----------------------------------------------------

* **Vision**: Tesla aims to become a top provider of autonomous solutions, competing in the developing market alongside traditional ride-hailing and taxi services. To achieve this goal, the company continues to make strategic investments in its FSD (Supervised) and neural network capabilities, while expanding its Robotaxi service globally. [Source: Page 6]
* **Expected Financial Impact**: By advancing a service-driven business model based on AI, software, and fleet-based profits, Tesla anticipates unlocking significant revenue opportunities. However, the company does not provide specific financial projections related to its Robotaxi service. [Source: Page 6]

Investor Implications
-----------------------

* **Positive Signals**: Tesla’s focus on autonomous driving technology and the launch of its Robotaxi service indicate the company’s commitment to innovation and long-term growth strategies. The potential for a service-driven business model based on AI, software, and fleet-based profits presents exciting opportunities for increased revenues and profitability.
* **Negative Signals**: The absence of concrete timelines, regulatory approvals, and financial projections raises questions about the viability and timing of Tesla’s Robotaxi ambitions. Growing competition in the autonomous driving and ride-hailing markets adds uncertainty to the company’s ability to capture market share and generate substantial returns.
* **Key Takeaways**: Investors should closely monitor Tesla’s progress in autonomous driving technology and the rollout of its Robotaxi service. Successful execution of these initiatives could drive significant growth and value creation for the company. Conversely, delays or setbacks could negatively impact Tesla’s financial performance and market position.

## Sources
- Tesla.pdf | Page 6
- Tesla.pdf | Page 140
- Tesla.pdf | Page 20
- Tesla.pdf | Page 54
- Tesla.pdf | Page 28

In [33]:
query = """Provide a detailed analysis of Tesla's Energy Generation and Storage segment.

Focus on:
- Powerwall and Megapack products and their applications
- Revenue growth, deployment volumes, and profitability trends
- Virtual power plants and software platforms (Powerhub, Autobidder)
- Competitive positioning vs traditional utilities and other energy storage companies
- Future growth outlook and strategic importance to Tesla

Explain each point above in very much depth and with intuition

Use exact figures and management commentary from the filings."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Provide a detailed analysis of Tesla's Energy Generation and...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 23/18 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 23/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7861 | Recall: 0.7860 | F1: 0.7861


# Financial Analysis — Tesla

Energy Generation and Storage Segment at Tesla
===============================================================

Tesla's Energy Generation and Storage (EGS) segment encompasses the production, sale, and leasing of energy storage solutions, including Powerwall and Megapack, as well as energy generation systems like solar panels and Solar Roof. These offerings aim to provide energy storage, generate renewable energy, and contribute to a more sustainable and efficient electric grid.

Revenue Growth and Deployment Volumes
--------------------------------------

The EGS segment experienced significant growth in revenue during the year ending December 31, 2025, increasing by $2.69 billion, or 27%. This expansion was driven primarily by increases in Megapack and Powerwall deployments, although there was a slight decrease in the average selling price of Megapack. Notably, this growth occurred despite the challenging economic environment and supply chain constraints faced by the industry.

Virtual Power Plants and Software Platforms
-------------------------------------------

Tesla leverages its expertise in artificial intelligence (AI) to enhance its energy storage products via firmware updates and software platforms. Key examples include Powerhub, which optimizes distributed energy resources like Powerwall for virtual power plant applications, and Autobidder, which controls and dispatches Megapack batteries within various market and application scenarios. These innovative software solutions enable Tesla to maximize the value of its energy storage assets, contributing to improved operational efficiency and overall profitability.

Competitive Positioning
-----------------------

Tesla's EGS segment competes with traditional utilities and specialized energy storage companies alike. Its unique advantage lies in the integration of its vehicles, energy storage products, and software platforms, creating a comprehensive suite of sustainable energy solutions. Furthermore, Tesla's focus on renewable energy sources, such as solar power, sets it apart from conventional energy providers reliant on fossil fuels.

Future Outlook and Strategic Importance
----------------------------------------

Looking ahead, Tesla aims to capitalize on the growing demand for sustainable energy solutions. The company plans to expand its EGS offerings, focusing on enhancing the efficiency and scalability of its Megapack and Powerwall products, as well as developing new energy generation systems. Moreover, Tesla intends to deepen its software capabilities, further optimizing its energy storage assets and expanding into new markets and applications.

In conclusion, Tesla's Energy Generation and Storage segment plays a crucial role in the company's long-term success. With a strong focus on innovation, sustainability, and integrated solutions, Tesla positions itself as a leading player in the global transition towards a carbon-neutral future.

## Sources
- Tesla.pdf | Page 7
- Tesla.pdf | Page 64
- Tesla.pdf | Page 9
- Tesla.pdf | Page 145
- Tesla.pdf | Page 66

In [34]:
query = """Analyze Tesla's supply chain, manufacturing operations, and associated risks.

Focus on:
- Dependency on key suppliers (especially battery cells)
- Manufacturing facilities and production ramp plans
- Geopolitical and raw material risks (lithium, nickel, etc.)
- Vertical integration efforts and in-house battery cell production
- Impact of potential disruptions on production and financials

Explain each point above in very much depth and with intuition


Assess overall supply chain vulnerability and mitigation strategies."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Analyze Tesla's supply chain, manufacturing operations, and ...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 19/17 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 19/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7934 | Recall: 0.7829 | F1: 0.7881


# Financial Analysis — Tesla

Tesla's Supply Chain, Manufacturing Operations, and Associated Risks
=========================================================================================

Tesla's supply chain involves thousands of parts sourced from hundreds of global suppliers, making it vulnerable to several potential risks. Key insights into the supply chain, manufacturing operations, and associated risks are discussed below.

Dependency on key suppliers
---------------------------

One of the main concerns for Tesla lies in its dependency on a few key suppliers, especially for battery cells. Currently, Tesla relies on suppliers like Panasonic and Contemporary Amperex Technology Co. Limited (CATL) for these essential components. This dependence limits Tesla's flexibility in changing suppliers and raises concerns about potential disruptions in the supply of battery cells. To address this issue, Tesla aims to supplement cells from its suppliers with cells manufactured by itself, believing that homegrown cells will be more efficient, scalable, and cost-effective in the long run. However, developing and manufacturing battery cells requires substantial investment, and there is no guarantee that Tesla will achieve these goals within the planned timeline or at all.

Manufacturing facilities and production ramp plans
--------------------------------------------------

Tesla operates manufacturing facilities in the United States (California, New York, Texas, and Nevada), China, and Germany. The company continues to expand production capacity at its existing facilities while striving to increase cost-competitiveness in major markets by strategically adding local manufacturing, often through partnerships with suppliers. However, expanding production capacity and meeting growing demand pose significant challenges for Tesla. Any production delays or inaccurate demand forecasting could negatively impact Tesla's business, financial condition, and operating results.

Geopolitical and raw material risks
------------------------------------

Raw material prices and availability play an important role in Tesla's supply chain. Prices for critical materials such as lithium, nickel, and other metals fluctuate and their availability may be affected by market conditions, trade policies, refining capacity, and global demand. For instance, increased global production of electric vehicles and energy storage products could strain the supply of raw materials. Furthermore, geopolitical events, such as trade policies, wars, and natural disasters, can disrupt the ability of Tesla's suppliers to deliver technologies or components to Tesla or to remain solvent and operational.

Vertical integration efforts and in-house battery cell production
------------------------------------------------------------------

To mitigate the risks associated with its dependence on external suppliers, Tesla is pursuing vertical integration by establishing an in-house lithium refinery in Texas. This move is aimed at localizing and de-risking the supply chain, but it remains to be seen how effective this strategy will be in addressing the challenges faced by Tesla's supply chain.

Impact of potential disruptions on production and financials
-------------------------------------------------------------

Potential disruptions in the supply of battery cells or other crucial components could limit Tesla's production of vehicles and energy storage products. Such disruptions could harm Tesla's business and operating results, as demonstrated by the impact of U.S. trade policy alterations in 2025, which led to heightened import tariffs and subsequent retaliatory measures affecting Tesla's supply chain costs.

Overall supply chain vulnerability and mitigation strategies
------------------------------------------------------------

Tesla's supply chain faces numerous vulnerabilities, including its heavy reliance on a few key suppliers, geopolitical risks, and the volatility of raw material prices. To mitigate these risks, Tesla is taking steps to diversify its supplier base, improve its internal manufacturing capabilities, and pursue vertical integration. Nevertheless, these efforts require substantial investment and carry no guarantees of success. It is essential for Tesla to continuously monitor its supply chain and adapt its strategies accordingly to minimize potential disruptions and maximize efficiency.

## Sources
- Tesla.pdf | Page 15
- Tesla.pdf | Page 32
- Tesla.pdf | Page 27
- Tesla.pdf | Page 25
- Tesla.pdf | Page 33

In [35]:
query = """Evaluate Tesla's competitive positioning and long-term corporate strategy.

Focus on:
- Competition in electric vehicles (traditional OEMs and new entrants)
- Competitive advantages in AI, software, and energy ecosystem
- Diversification into Robotaxi, Optimus Bots, and Energy
- Capital allocation priorities (R&D, capex, buybacks)
- Risks from new competitors and regulatory changes

Explain each point above in very much depth and with intuition


Conclude with overall strategic strengths, weaknesses, and investor implications."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Evaluate Tesla's competitive positioning and long-term corpo...
   Retrieval Quality: 14/17 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 14/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7387 | Recall: 0.7432 | F1: 0.7410


# Financial Analysis — Tesla

Tesla's Competitive Positioning and Long-Term Corporate Strategy
===============================================================================

### Electric Vehicles Market Overview

In the highly competitive global automotive market, Tesla faces competition from both traditional Original Equipment Manufacturers (OEMs) and emerging players entering the electric vehicle (EV) space. These competitors include internal combustion engine (ICE) carmakers transitioning to EVs, as well as startups focusing solely on EVs. Examples of notable competitors include General Motors, Ford Motor Company, Volkswagen Group, NIO, Lucid Motors, and Rivian Automotive.

#### Traditional OEMs

Traditional OEMs have substantial resources dedicated to designing, developing, manufacturing, distributing, promoting, selling, and supporting their products and services. They often achieve cost efficiencies due to their established locations and economic environments. This increased competition could potentially lead to lower vehicle unit sales, price reductions, revenue shortfalls, lost customers, and lost market share, which may harm Tesla's business, financial condition, and operating results.

#### Emerging Players

New EV entrants, such as NIO, Lucid Motors, and Rivian Automotive, pose a threat to Tesla's market share. These companies are investing heavily in research and development (R&D), manufacturing capabilities, and marketing efforts to challenge Tesla's dominance in the EV market. However, Tesla's early mover advantage, brand recognition, and superior technological capabilities give it a competitive edge in this landscape.

### Competitive Advantages in AI, Software, and Energy Ecosystem

Tesla leverages its strength in artificial intelligence (AI), software, and energy ecosystem to maintain a competitive edge.

#### AI and Autonomous Driving Technology

Tesla's advanced autonomous driving technology sets it apart from competitors. Its Full Self-Driving (FSD) capability provides a unique selling proposition, enabling features such as automatic lane changing, navigation on city streets, and parking spot detection. While other OEMs are working on similar technologies, Tesla's FSD has already been deployed in several thousand vehicles, giving it a significant advantage in this area.

#### Supercharger Network and Energy Storage Solutions

Tesla's extensive Supercharger network offers convenience to EV owners, providing fast charging solutions along popular travel routes. Additionally, Tesla's energy storage solutions, such as Powerwalls and Megapacks, allow customers to store excess solar power generated during the day for later use. This integrated energy ecosystem creates a stickier relationship between Tesla and its customers, further solidifying its competitive position.

#### Software Updates and Over-the-Air (OTA) Capabilities

Tesla's ability to deliver software updates over-the-air (OTA) enables continuous improvement of its vehicles without requiring physical dealership visits. This feature enhances the user experience, increases customer satisfaction, and reduces service costs. Other OEMs are beginning to adopt similar strategies, but Tesla remains a pioneer in this area.

### Diversification into Robotaxis, Optimus Bots, and Energy

Tesla is expanding beyond electric vehicles into robotaxis, Optimus Bots, and energy solutions.

#### Robotaxis

Tesla's robotaxi initiative aims to leverage its existing fleet of vehicles to create a ride-hailing service powered by autonomous vehicles. By offering a safe, convenient, and affordable transportation option, Tesla hopes to capture a significant portion of the ride-hailing market. However, the success of this venture depends on the development and commercialization of its autonomous driving technology.

#### Optimus Bots

Optimus, Tesla's humanoid robot designed for household chores and heavy lifting tasks, represents another avenue for diversification. Although still in the development stage, Optimus has the potential to revolutionize industries such as agriculture, construction, and logistics. Successfully bringing Optimus to market would establish Tesla as a leader in the burgeoning field of robotic domestic assistance.

#### Energy Solutions

Tesla's energy solutions, including solar energy systems, Powerwalls, and Megapacks, provide a complementary source of revenue and strengthen the company's position as a sustainable energy provider. By integrating solar power, battery storage, and electric vehicles, Tesla offers customers a complete energy ecosystem that promotes energy independence and sustainability.

### Capital Allocation Priorities

Tesla's capital allocation priorities focus on R&D, capital expenditures (capex), share repurchases, and dividends.

#### R&D

Tesla dedicates significant resources to R&D, investing in cutting-edge technologies such as autonomous driving, battery technology, and robotics. This investment is crucial for staying ahead of competitors and maintaining a competitive edge in the rapidly evolving EV market.

#### Capex

Capital expenditures are directed towards expanding manufacturing capacity, improving production efficiency, and building new facilities. For instance, Tesla has invested in Gigafactories to produce batteries at scale and reduce costs. These investments are essential for meeting increasing demand for EVs and maintaining a competitive manufacturing footprint.

#### Share Repurchases

Tesla has a history of buying back shares, reducing the number of outstanding shares and increasing earnings per share (EPS). This strategy returns value to shareholders and supports the company's stock price.

#### Dividends

Unlike many traditional OEMs, Tesla does not pay dividends. Instead, it prioritizes reinvesting profits to fund R&D, capex, and share repurchases. This decision reflects Tesla's commitment to long-term growth and innovation rather than short-term gains.

### Risks from New Competitors and Regulatory Changes

Tesla faces risks from new competitors entering the EV market and regulatory changes affecting its business.

#### New Competitors

As more players join the EV market, competition intensifies, potentially leading to lower vehicle unit sales, price reductions, revenue shortfalls, lost customers, and lost market share. Tesla's ability to adapt quickly and continue innovating will be critical for maintaining its competitive edge.

#### Regulatory Changes

Regulatory changes, such as changes to net metering policies, tariff structures, and subsidy programs, can impact Tesla's energy generation and storage business. Adapting to these changes requires ongoing monitoring and adjusting strategies accordingly to ensure continued success in these areas.

### Strategic Strengths, Weaknesses, and Investor Implications

Tesla's strategic strengths lie in its early mover advantage, brand recognition, superior technological capabilities, and integrated energy ecosystem. These factors give it a competitive edge in the EV market and set it apart from traditional OEMs and emerging players.

However, Tesla also has weaknesses, including its reliance on autonomous driving technology, vulnerability to regulatory changes, and exposure to cyclical commodity prices. Addressing these weaknesses will require careful management and strategic planning.

For investors, Tesla presents an exciting opportunity due to its leadership role in the EV market, commitment to innovation, and diverse revenue streams. However, the company's aggressive growth strategy, reliance on autonomous driving technology, and exposure to regulatory changes create inherent risks that should be carefully considered before investing.

Overall, Tesla's competitive positioning and long-term corporate strategy demonstrate a clear focus on innovation, diversification, and sustainable growth. By continuing to invest in R&D, capex, and strategic initiatives, Tesla positions itself for long-term success in the EV market and beyond.

## Sources
- Tesla.pdf | Page 21
- Tesla.pdf | Page 33
- Tesla.pdf | Page 15
- Tesla.pdf | Page 28
- Tesla.pdf | Page 20
- Tesla.pdf | Page 55

In [36]:
print(financial_rag._last_context)

---
type: FinancialText
company: Tesla
source_file: Tesla.pdf
page: 21
---

Table of Contents
Energy Generation Systems
The primary competitors to our energy generation business are the traditional local utility companies that supply energy to our
potential customers. We compete with these traditional utility companies primarily based on price and the ease by which customers can
switch to electricity generated by our energy generation systems. We also compete with solar energy companies that provide products
and services similar to ours. Many solar energy companies only install solar energy systems, while others only provide financing for
these installations. We believe we have a significant expansion opportunity with our offerings, including in terms of the aesthetics,
superior performance and ease of installation and integration with Powerwall of our solar panels, and that the environment is
increasingly conducive to the adoption of renewable energy systems. Intellectual Property
We 